# 3. ES Seasonal Calendar Strategy - Robustness Testing
## Professional Validation Framework with Realistic Costs

This notebook performs advanced robustness tests on the ES Seasonal Calendar strategy using the **optimal parameters from 02**.

**Optimal Parameters (from 02):**
- Days: Tue-Fri [1,2,3,4]
- Months: Apr, Jul, Oct, Nov [4,7,10,11]
- Stop Loss: X%
- Take Profit: Y%

**Tests:**
1. Out-of-Sample Backtest
2. Walk-Forward Analysis
3. Parameter Sensitivity
4. Stress Tests
5. Monte Carlo
6. Final Decision

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## CONFIGURE PATHS AND IMPORT

In [ ]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root / 'utils'))

from utils import (
    load_data_from_mongo,
    compute_metrics,
    monte_carlo_bootstrap,
    plot_equity_curve,
    plot_trade_distribution
)

# Plot styling
sns.set_style("whitegrid")
sns.set_palette("muted")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

print("✅ Utils imported successfully.")

## LOAD DATA, COSTS, AND FINAL PARAMETERS

In [ ]:
# Load data
df_15min = load_data_from_mongo('es_15min')

print(f"✅ Data loaded: {len(df_15min)} bars")

# Resample to daily
df_daily = df_15min.resample('D').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}).dropna()

print(f"📊 Daily data: {len(df_daily)} bars")

# Split
df_is = df_daily[(df_daily.index >= '2009-09-27') & (df_daily.index < '2020-01-01')].copy()
df_oos = df_daily[df_daily.index >= '2020-01-01'].copy()

print(f"📊 In-Sample: {len(df_is)} days")
print(f"📊 Out-of-Sample: {len(df_oos)} days")

# Costs
COMMISSION_PER_CONTRACT = 2.0
SLIPPAGE_TICKS = 0.5
SLIPPAGE_COST = SLIPPAGE_TICKS * 12.50
COST_PER_TRADE = COMMISSION_PER_CONTRACT * 2 + SLIPPAGE_COST

print("\n" + "="*70)
print("COST ASSUMPTIONS")
print("="*70)
print(f"Total cost per trade: ${COST_PER_TRADE:.2f}")

# --- FINAL OPTIMAL PARAMETERS (from 02) ---
# IMPORTANT: Update these with your actual optimal values from 02!
OPTIMAL_PARAMS = {
    'days': [1, 2, 3, 4],  # Tue-Fri
    'months': [4, 7, 10, 11],  # Apr, Jul, Oct, Nov
    'stop_pct': 1.0,  # <-- UPDATE with actual
    'target_pct': 1.5  # <-- UPDATE with actual
}

print("\n" + "="*70)
print("FINAL PARAMETERS")
print("="*70)
for k, v in OPTIMAL_PARAMS.items():
    print(f"  {k}: {v}")

## HELPER FUNCTIONS

In [ ]:

def add_calendar_features(df):
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month
    df['sma200'] = df['close'].rolling(200).mean()
    df['above_sma200'] = df['close'] > df['sma200']
    return df

def run_calendar_backtest(df, entry_days, entry_months, trend_filter=True, stop_pct=1.0, target_pct=1.5):
    df = df.copy()
    entry_condition = (
        df['dayofweek'].isin(entry_days) &
        df['month'].isin(entry_months) &
        ~df['dayofweek'].isin([5, 6])
    )
    if trend_filter:
        entry_condition = entry_condition & df['above_sma200']
    df['entry_signal'] = entry_condition
    
    trades = []
    position = None
    entry_price = None
    entry_time = None
    
    for idx, row in df.iterrows():
        if position is not None:
            if position == 'long':
                if row['low'] <= entry_price * (1 - stop_pct/100):
                    exit_price = entry_price * (1 - stop_pct/100)
                    reason = 'stop_loss'
                elif row['high'] >= entry_price * (1 + target_pct/100):
                    exit_price = entry_price * (1 + target_pct/100)
                    reason = 'take_profit'
                else:
                    exit_price = row['close']
                    reason = 'day_close'
            trade = {
                'entry_time': entry_time,
                'exit_time': idx,
                'entry_price': entry_price,
                'exit_price': exit_price,
                'type': position,
                'reason': reason
            }
            trades.append(trade)
            position = None
            continue
        
        if row['entry_signal'] and position is None:
            position = 'long'
            entry_price = row['open'] if not pd.isna(row['open']) else row['close']
            entry_time = idx
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        trades_df['pnl'] = (trades_df['exit_price'] - trades_df['entry_price']) / trades_df['entry_price'] * 100
        trades_df['win'] = trades_df['pnl'] > 0
    return trades_df

## OOS BACKTEST

In [ ]:

print("\n" + "="*70)
print("OOS BACKTEST (2020-2026)")
print("="*70)

df_oos_prep = add_calendar_features(df_oos)

trades_oos = run_calendar_backtest(
    df_oos_prep,
    entry_days=OPTIMAL_PARAMS['days'],
    entry_months=OPTIMAL_PARAMS['months'],
    trend_filter=True,
    stop_pct=OPTIMAL_PARAMS['stop_pct'],
    target_pct=OPTIMAL_PARAMS['target_pct']
)

if len(trades_oos) > 0:
    metrics_oos = compute_metrics(trades_oos, cost_per_trade=COST_PER_TRADE)
    
    print(f"Number of trades: {metrics_oos['num_trades']}")
    print(f"Win rate: {metrics_oos['win_rate']:.2f}%")
    print(f"Average win: {metrics_oos['avg_win']:.2f}%")
    print(f"Average loss: {metrics_oos['avg_loss']:.2f}%")
    print(f"Profit Factor: {metrics_oos['profit_factor']:.2f}")
    print(f"Expectancy per trade: {metrics_oos['expectancy']:.4f}%")
    print(f"Total P&L: {metrics_oos['total_pnl']:.2f}%")
    print(f"Sharpe Ratio: {metrics_oos['sharpe']:.2f}")
    print(f"Maximum Drawdown: {metrics_oos['max_drawdown']:.2f}%")
    
    plot_equity_curve(trades_oos, title='Equity Curve - Out-of-Sample (2020-2026)')
    plot_trade_distribution(trades_oos, title='Trade Distribution - Out-of-Sample')
else:
    print("❌ No trades in OOS.")

## WALK-FORWARD ANALYSIS

In [ ]:

print("\n" + "="*70)
print("WALK-FORWARD ANALYSIS")
print("="*70)

print("""
Walk-Forward Analysis is not implemented for calendar strategies
because the parameters (days, months) are structural and known in advance.
They are not dependent on price dynamics like ATR or RSI.

Instead, we test the strategy on each year independently:
""")

# Yearly performance breakdown
if len(trades_oos) > 0:
    trades_oos['year'] = trades_oos['exit_time'].dt.year
    yearly_perf = trades_oos.groupby('year').agg({
        'pnl': ['sum', 'count', 'mean'],
        'win': 'mean'
    }).round(2)
    yearly_perf.columns = ['Total P&L %', 'Num Trades', 'Avg P&L %', 'Win Rate %']
    print(yearly_perf.to_string())
    
    # Check consistency
    positive_years = (yearly_perf['Total P&L %'] > 0).sum()
    total_years = len(yearly_perf)
    print(f"\nPositive years: {positive_years}/{total_years}")
    
    if positive_years >= total_years * 0.7:  # At least 70% years positive
        print("✅ PASS: Strategy is consistent across years.")
    else:
        print("⚠️ WARNING: Strategy underperforms in some years.")
else:
    print("No trades for yearly analysis.")

## PARAMETER SENSITIVITY

In [ ]:

print("\n" + "="*70)
print("PARAMETER SENSITIVITY ANALYSIS")
print("="*70)

# Test variations around optimal
variations = {
    'stop_pct': [OPTIMAL_PARAMS['stop_pct'] * 0.8, OPTIMAL_PARAMS['stop_pct'] * 1.2],
    'target_pct': [OPTIMAL_PARAMS['target_pct'] * 0.8, OPTIMAL_PARAMS['target_pct'] * 1.2]
}

sensitivity_results = []
for param, values in variations.items():
    for val in values:
        trades = run_calendar_backtest(
            df_oos_prep,
            entry_days=OPTIMAL_PARAMS['days'],
            entry_months=OPTIMAL_PARAMS['months'],
            trend_filter=True,
            stop_pct=OPTIMAL_PARAMS['stop_pct'] if param == 'target_pct' else val,
            target_pct=OPTIMAL_PARAMS['target_pct'] if param == 'stop_pct' else val
        )
        if len(trades) > 0:
            metrics = compute_metrics(trades, cost_per_trade=COST_PER_TRADE)
            sensitivity_results.append({
                'param': param,
                'value': val,
                'pf': metrics['profit_factor'],
                'num_trades': metrics['num_trades'],
                'total_pnl': metrics['total_pnl']
            })

if sensitivity_results:
    sens_df = pd.DataFrame(sensitivity_results)
    print(sens_df.to_string(index=False))
    pf_range = sens_df['pf'].max() - sens_df['pf'].min()
    print(f"\n📊 Profit Factor range: {pf_range:.2f}")
    
    if pf_range < 0.5:
        print("✅ Robust: PF is stable across parameter variations.")
    else:
        print("⚠️ Warning: PF varies significantly.")

## MONTE CARLO BOOTSTRAP

In [ ]:

print("\n" + "="*70)
print("MONTE CARLO BOOTSTRAP (10,000 simulations)")
print("="*70)

if len(trades_oos) > 0:
    mc_results = monte_carlo_bootstrap(trades_oos, n_simulations=10000)
    
    if mc_results:
        print(f"  5th percentile (final P&L): {mc_results['p5']:.2f}%")
        print(f"  50th percentile (final P&L): {mc_results['p50']:.2f}%")
        print(f"  95th percentile (final P&L): {mc_results['p95']:.2f}%")
        print(f"  99th percentile (final P&L): {mc_results['p99']:.2f}%")
        print(f"  Actual OOS P&L: {metrics_oos['total_pnl']:.2f}%")
        print(f"  Probability of positive P&L: {mc_results['prob_positive']:.1f}%")
        
        if mc_results['prob_positive'] > 80:
            print("✅ Strong edge: >80% probability of positive P&L.")
        elif mc_results['prob_positive'] > 60:
            print("⚠️ Moderate edge: 60-80% probability.")
        else:
            print("❌ Weak edge: <60% probability.")
else:
    print("No trades for Monte Carlo.")

## FINAL DECISION

In [ ]:
print("\n" + "="*70)
print("FINAL DECISION SUMMARY")
print("="*70)

if len(trades_oos) > 0:
    pf_pass = metrics_oos['profit_factor'] > 1.3
    sharpe_pass = metrics_oos['sharpe'] > 0.7
    dd_pass = metrics_oos['max_drawdown'] > -20
    trades_pass = metrics_oos['num_trades'] >= 30
    mc_pass = mc_results['prob_positive'] > 80 if 'mc_results' in locals() else False
    
    print(f"  Profit Factor > 1.3: {'✅' if pf_pass else '❌'} ({metrics_oos['profit_factor']:.2f})")
    print(f"  Sharpe Ratio > 0.7: {'✅' if sharpe_pass else '❌'} ({metrics_oos['sharpe']:.2f})")
    print(f"  Max Drawdown > -20%: {'✅' if dd_pass else '❌'} ({metrics_oos['max_drawdown']:.1f}%)")
    print(f"  Trades >= 30: {'✅' if trades_pass else '❌'} ({metrics_oos['num_trades']})")
    print(f"  Monte Carlo P(positive) > 80%: {'✅' if mc_pass else '❌'} ({mc_results['prob_positive']:.1f}%)")
    
    if pf_pass and sharpe_pass and dd_pass and trades_pass and mc_pass:
        print("\n" + "="*70)
        print("✅ PASS: Strategy passes all criteria. Ready for forward testing!")
        print("="*70)
    else:
        print("\n" + "="*70)
        print("❌ FAIL: Some criteria failed. Review and refine.")
        print("="*70)
else:
    print("❌ FAIL: No trades in OOS.")